# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides users in loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. All dataset elements—record sets, fields, and columns—are referenced by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL for FAIR^2
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we print all record sets available in the package by their `@id`, listing their human-friendly names and included fields, all referenced by `@id`. This gives you a quick sense of the structure and dimension of the dataset.

In [ ]:
# Show all record sets, fields, and columns by @id
print("Available Record Sets (@id):\n")
record_set_ids = []
for record_set in metadata.record_sets:
    print(f"- @id: {record_set['@id']}")
    record_set_ids.append(record_set['@id'])
    print(f"  Name:  {record_set.get('name','')}  | Description: {record_set.get('description','')}")
    print("  Fields:")
    if 'fields' in record_set:
        for field in record_set['fields']:
            print(f"    - @id: {field['@id']} | name: {field.get('name', '')} | dataType: {field.get('dataType','')}")
    print()
if not record_set_ids:
    print('No record_sets were automatically listed in metadata. Attempting to find by using dataset.record_sets property...')
    # In case the schema did not expand record_sets, infer from dataset internals
    inferred = []
    try:
        for rs in dataset.record_sets:
            print(f"- @id: {rs['@id']}")
            inferred.append(rs['@id'])
    except Exception:
        pass
    record_set_ids = inferred

## Inspecting the Records
Next, we print a sample record from each record set using their `@id`. This helps confirm the presence and schema of the actual records retrieved (all keys are field `@id`s):

In [ ]:
# For each record set, print the first record
for rs_id in record_set_ids:
    print(f"\nSample record from record set: {rs_id}")
    try:
        sample = next(dataset.records(record_set=rs_id))
        for k, v in sample.items():
            print(f"  {k}: {v}")
    except Exception as e:
        print("  <no records or error>", e)

## 3. Data Extraction
Load all records from each record set into a DataFrame for analysis.

We will create a dictionary `dataframes` mapping each record set `@id` to its corresponding table (DataFrame), with field columns always referenced by `@id`.

In [ ]:
# Extract records from each record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set {rs_id}")
    else:
        print(f"No records found for record set {rs_id}")

# Select a main record set to demo columns and head()
if dataframes:
    primary_rs_id = list(dataframes.keys())[0]
    print(f"\nPrimary record set chosen for demonstration: {primary_rs_id}")
    print("Columns (field @id):", dataframes[primary_rs_id].columns.tolist())
    display(dataframes[primary_rs_id].head())
else:
    print('No data loaded. Please check record_set_ids and records.')

## 4. Exploratory Data Analysis (EDA)
This section demonstrates typical data processing:
- Filtering records based on a numeric field (`@id` reference)
- Normalizing a numeric column
- Grouping and aggregating data using categorical fields (by `@id`)

Ensure to update the variables `numeric_field_id` and `group_field_id` below to match an existing field `@id` from your DataFrame. The first numeric field and categorical field found in the schema will be used for demonstration.

In [ ]:
# Identify a numeric field and group field from metadata
import re
numeric_field_id = None
group_field_id = None

if primary_rs_id in dataframes:
    df = dataframes[primary_rs_id]
    # Guess some numeric and categorical fields by inspecting the columns
    # (ideally we would use schema but fallback to string heuristics)
    for f in df.columns:
        if re.search(r'age|interval|year|count|number|duration|score|size|distance|percent|total|length|level', f, re.I):
            # Check for numeric content
            try:
                if pd.api.types.is_numeric_dtype(df[f].astype(float)):
                    numeric_field_id = f
                    break
            except Exception:
                pass

    # Now guess a group field: prefer sex, gender, anatomical site, MSI status, etc.
    for f in df.columns:
        if re.search(r'sex|gender|site|location|anatomical|MSI|status|type|group|histology', f, re.I):
            if pd.api.types.is_object_dtype(df[f]):
                group_field_id = f
                break

    print(f"Numeric field for EDA: {numeric_field_id}")
    print(f"Grouping field for EDA: {group_field_id}")

    # Try to coerce numeric if not already
    if numeric_field_id:
        df_numeric = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df_numeric.mean()  # Use the mean as an example threshold
        filtered_df = df[df_numeric > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df[[numeric_field_id]].head())

        # Normalization
        filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id].astype(float) - df_numeric.mean()) / df_numeric.std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())
    else:
        print('No numeric field detected for EDA.')

    # Grouping
    if group_field_id and numeric_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print('No main DataFrame for EDA.')

## 5. Visualization
Visualize the distribution of the numeric variable and explore potential group differences with respect to the group field.

_All axes and legends use field `@id`s to ensure future reproducibility._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_rs_id in dataframes and numeric_field_id and group_field_id:
    fdf = dataframes[primary_rs_id].copy()
    # Filter out nan for plotting
    fdf[numeric_field_id] = pd.to_numeric(fdf[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8, 4))
    sns.histplot(fdf[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=fdf)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30, ha='right')
    plt.show()
else:
    print('Cannot plot: Data or required fields not available.')

## 6. Conclusion
In this notebook, we demonstrated how to programmatically discover and reference record sets, fields, and columns by their `@id` in a FAIR² Croissant dataset. 

- We loaded and explored the dataset schema, printing out all available entities by unique ID.
- We loaded record data into DataFrames, referencing all fields and grouping variables using their `@id`.
- We performed filtering, normalization, aggregation, and visualization by field `@id`s, preserving full reproducibility for FAIR workflows.

This workflow can be adapted to any Croissant-compliant dataset for robust and reproducible data science.